# MM4250 stuck check

Steps the switch through ALL_OPEN, internal short, ALL_OPEN, internal load, ALL_OPEN, then RF1, ALL_OPEN, RF2, ALL_OPEN ... RF6, ALL_OPEN. For every position it saves:

- the raw sweep into the lab's stuck-check database, `<project>_Cooldown_<NN>_stuck.db` (turn off with `SAVE_DB = False`),
- one raw `.s2p` plus `positions.csv` (time, run id, and the fridge's MXP, MXP2 and STP readings just before and just after each position; also in the database) in `Sweeps\stuck_check\<date>_<time>_<serial>_<temp>\`,

and then plots the result (also saved there as `stuck_check.png`). Everything goes in this cooldown's folder, `E:\QCoDeS\<project>\Cooldown_<NN>\`, beside the database: the database, `Sweeps\` and `db_backups\` (checked copies of the database).

- Run the cells top to bottom, once each.
- **First close `mm4250_sweeps.ipynb`** (its Close cell, or restart its kernel): only one notebook can hold the switch.
- VNA port 1 on fridge line RF6, port 2 on line RF2. **The LNA must be on** (4 K bias when cold), or nothing comes back and every position looks the same.


In [ ]:
import sys, csv
from datetime import datetime
from pathlib import Path
import numpy as np

try:
    here = Path(_dh[0])      # this notebook's folder (Jupyter)
except NameError:
    here = Path.cwd()
root = here
while not (root / "drivers").is_dir() and root != root.parent:
    root = root.parent       # the repo with drivers/ in it
sys.path[:0] = [str(here), str(root)]

from drivers.KeysightVNA_driver import KeysightP5004B
from drivers.MM4250_QCodes_driver import MM4250
from vna_measure import setup_sweep, measure_2port   # moves the switch with the VNA source off
from sweep_db import _leave_switch_open, record_measurement, _open_experiment, _checkpoint, _safe_backup
from sweep_db import find_setup_database, _lab_db   # the lab's database helper (scripts/database_helper.py)
from read_db import save_touchstone
from plots import plot_stuck_check, stuck_check_folders

%matplotlib inline

**Settings.** Normally only `COOLDOWN_NUM` changes (once per cooldown). The temperature in the folder name comes from the fridge; if the fridge isn't connected, type it in `TEMP_LABEL`.

In [ ]:
TEMP_LABEL = None                   # None = from the fridge (mixing plate); or type one, e.g. "60K"
SERIAL = "SN0077"
CHANNELS = [1, 2, 3, 4, 5, 6]       # switch channels to check
SWEEP = dict(start=510e6, stop=610e6, points=2001, if_bandwidth=1e3, power=-20)   # circulator band, -20 dBm
SAVE_DB = True                      # True: every position into the lab's stuck-check database
COOLDOWN_NUM = 5                    # the fridge's cooldown number: change it every cooldown
DB_PROJECT = "MM4250"               # -> E:\QCoDeS\MM4250\Cooldown_05\MM4250_Cooldown_05_stuck.db


**Where things go.** Works out this cooldown's folder from the lab's database helper (nothing is made yet) and prints it. No instruments needed.

In [ ]:
try:
    setup_database = find_setup_database()                      # the lab's helper, scripts/database_helper.py
    DB_FILE = Path(sys.modules[setup_database.__module__].database_path(DB_PROJECT, COOLDOWN_NUM, "stuck"))
except (FileNotFoundError, ImportError) as e:
    setup_database = None
    DB_FILE = here / "local_databases" / f"{DB_PROJECT}_Cooldown_{COOLDOWN_NUM:02d}_stuck.db"
    print(f"WARNING: no lab database helper ({e}).\n"
          "  Everything goes into local_databases\\ beside this notebook instead. On the DAQ this means something is wrong: stop and check.")
DB_FILE = DB_FILE.resolve()
COOLDOWN_DIR = DB_FILE.parent                                    # database, Sweeps\ and db_backups\ all go here
print(f"Cooldown folder: {COOLDOWN_DIR}")
print(f"  database:  {DB_FILE.name}" + ("" if SAVE_DB else "   (not used: SAVE_DB = False)"))
print(f"  sweeps:    Sweeps\\stuck_check\\")


**Connect.** Connecting the switch sets ALL_OPEN. The fridge is only for temperatures; if it doesn't connect (no VPN), the check still runs. Then it opens the database in the cooldown folder from the cell above.

In [ ]:
from qcodes.instrument import Instrument
Instrument.close_all()              # so re-running this cell is safe

ksvna = KeysightP5004B("ksvna", "TCPIP0::QTSF-Measurement::hislip_PXI0_CHASSIS1_SLOT1_INDEX0::INSTR")
switch = MM4250("switch")

try:
    from maybell_qcodes import MaybellFridge
    fridge = MaybellFridge("fridge", host="10.179.200.168")
except Exception as e:
    fridge = None
    print(f"No fridge temperatures ({type(e).__name__}: {e}). The check still runs.")

SENSORS = ("mxp", "mxp2", "stp")    # the fridge's MXP, MXP2 and STP thermometers

def fridge_temps():
    """{sensor: kelvin or None} for SENSORS. None = no fridge, an error, or no calibration curve."""
    out = {}
    for s in SENSORS:
        try:
            value = getattr(fridge, s).temperature() if fridge is not None else None
            out[s] = float(value) if value is not None and float(value) > 0 else None
        except Exception:
            out[s] = None
    return out

def mxp_kelvin():
    """Mixing plate temperature in K, or None if there's no reading."""
    return fridge_temps()["mxp"]

db_path, exp = None, None
if SAVE_DB:
    db_path = _lab_db(setup_database, DB_PROJECT, COOLDOWN_NUM, "stuck") if setup_database else DB_FILE
    exp, db_path = _open_experiment(SERIAL, db_path, "stuck_check")
else:
    print("SAVE_DB = False: files only, nothing goes into a database.")


**Run.** About a minute. At the end it prints a table and draws the plot (saved in the folder as `stuck_check.png`). Reading it (compare neighbouring rows, not fixed numbers):

- Every blue bar tall, every grey dot back near 0: **all good**.
- A blue bar near 0: that position **didn't close** (stuck open).
- A grey dot that stays high: the position before it **didn't open** (stuck closed).

If anything is stuck, stop: don't keep switching it cold. Run the Close cell and tell Charlie. The switch is always left ALL_OPEN at the end, even if this cell fails.

In [ ]:
seq = ["ALL_OPEN", "INTERNAL_SHORT", "ALL_OPEN", "INTERNAL_LOAD", "ALL_OPEN"]
for ch in CHANNELS:
    seq += [ch, "ALL_OPEN"]

t0 = mxp_kelvin()
label = TEMP_LABEL or (f"{t0:.0f}K" if t0 is not None and t0 >= 1 else f"{t0 * 1e3:.0f}mK" if t0 is not None else "noT")
out = COOLDOWN_DIR / "Sweeps" / "stuck_check" / f"{datetime.now():%Y%m%d_%H%M%S}_{SERIAL}_{label}"
setup_sweep(vna=ksvna, **SWEEP)
ksvna.write("SENS:CORR:CSET:DEAC")  # raw data: VNA calibration off

rows, ref = [], None
try:
    for i, pos in enumerate(seq, 1):
        name = f"RF{pos}" if isinstance(pos, int) else pos
        before = fridge_temps()         # every thermometer just before and just after the sweep
        if isinstance(pos, int):
            freq, data = measure_2port(pos, vna=ksvna, switch=switch)
        else:
            freq, data = measure_2port(state=pos, vna=ksvna, switch=switch)
        after = fridge_temps()
        temps = {f"{s}_k_{w}": t[s] for w, t in (("before", before), ("after", after)) for s in SENSORS}
        temp_k = before["mxp"]
        path = out / f"{i:02d}_{name}.s2p"
        save_touchstone(freq, data, path)
        if ref is None:
            ref = data["S21"]       # the first ALL_OPEN
        diff = float(np.median(np.abs(data["S21"] - ref)))
        run_id = None
        if db_path is not None:     # the same sweep into the database, as soon as it's measured
            run_id = record_measurement(name, freq, data, exp, touchstone_path=path,
                                        check_id=out.name, check_type="stuck", position=name, step=i,
                                        switch_serial=SERIAL, temp_str=label, mxc_temp_k=temp_k,
                                        cooldown=COOLDOWN_NUM, diff_from_first_open=diff, **temps).run_id
        rows.append([i, name, path.name, datetime.now().isoformat(timespec="seconds"), temp_k, diff, run_id,
                     *temps.values()])
finally:
    _leave_switch_open(switch, ksvna)   # ALL_OPEN, even after an error
    if rows:
        with open(out / "positions.csv", "w", newline="") as f:
            w = csv.writer(f)
            w.writerow(["step", "position", "file", "time", "mxp_K", "median_abs_S21_minus_first_open", "run_id",
                        *[f"{s}_k_{w}" for w in ("before", "after") for s in SENSORS]])
            w.writerows(rows)
    if db_path is not None:
        _checkpoint(db_path)            # newest runs into the .db file itself
        _safe_backup(db_path)           # a checked copy in db_backups/ beside it

print(f"\nSaved {len(rows)} files to {out}\n")
for i, name, _, _, temp_k, diff, *_ in rows:
    t = "no reading" if temp_k is None else f"{temp_k * 1e3:.1f} mK" if temp_k < 1 else f"{temp_k:.2f} K"
    print(f"{i:02d}  {name:>14}   difference from first ALL_OPEN: {diff:.3g}   ({t})")
if db_path is not None:
    print(f"\nDatabase: {db_path}, runs {rows[0][6]}-{rows[-1][6]} (check_id {out.name})")

plot_stuck_check(out)


**Plot an earlier check.** Needs only the first cells up to *Where things go* (no instruments). `FOLDER = None` plots the newest one in this cooldown's `Sweeps\stuck_check\`; or give a folder name, e.g. `"20261111_214725_SN0077_92mK"`. Saves another `stuck_check.png` (never overwrites).

In [ ]:
FOLDER = None

folders = stuck_check_folders(COOLDOWN_DIR)
print("\n".join(p.name for p in folders) or "no stuck checks in Sweeps/stuck_check/ yet")
if folders:
    plot_stuck_check(folders[-1] if FOLDER is None else COOLDOWN_DIR / "Sweeps" / "stuck_check" / FOLDER)

**Close.** Run when done, before anyone else uses the switch or VNA. Also backs up the database.

In [ ]:
ksvna.output(False)
switch.open_all()
ksvna.close()
switch.close()
if fridge is not None:
    fridge.close()
if db_path is not None:
    _safe_backup(db_path)
